In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
df = pd.read_csv("data/raw/BankChurners.csv")

print("Formato do dataset:", df.shape)
df.info()
df.head()

Formato do dataset: (10127, 23)
<class 'pandas.DataFrame'>
RangeIndex: 10127 entries, 0 to 10126
Data columns (total 23 columns):
 #   Column                                                                                                                              Non-Null Count  Dtype  
---  ------                                                                                                                              --------------  -----  
 0   CLIENTNUM                                                                                                                           10127 non-null  int64  
 1   Attrition_Flag                                                                                                                      10127 non-null  str    
 2   Customer_Age                                                                                                                        10127 non-null  int64  
 3   Gender                                                      

,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0.000134,0.99987
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0.000022,0.99998


In [4]:
print("Linhas duplicadas (todas as colunas):", df.duplicated().sum())
print("CLIENTNUM duplicados:", df['CLIENTNUM'].duplicated().sum())

print("\nValores nulos reais (NaN) no arquivo bruto:", df.isnull().sum().sum())

for col in ['Education_Level', 'Marital_Status', 'Income_Category']:
    qtd_unknown = (df[col] == 'Unknown').sum()
    print(f"'{col}' com 'Unknown': {qtd_unknown} ({qtd_unknown/len(df)*100:.1f}%)")


print("\nTop 5 valores mais repetidos em Credit_Limit:")
print(df['Credit_Limit'].value_counts().head(5))

diferenca = (df['Credit_Limit'] - df['Total_Revolving_Bal']) - df['Avg_Open_To_Buy']
print(f"\nDiferença máxima entre Avg_Open_To_Buy e (Credit_Limit - Total_Revolving_Bal): {diferenca.abs().max():.2e}")
print("-> Confirmado: Avg_Open_To_Buy é redundante e será removida.")

Linhas duplicadas (todas as colunas): 0
CLIENTNUM duplicados: 0

Valores nulos reais (NaN) no arquivo bruto: 0
'Education_Level' com 'Unknown': 1519 (15.0%)
'Marital_Status' com 'Unknown': 749 (7.4%)
'Income_Category' com 'Unknown': 1112 (11.0%)

Top 5 valores mais repetidos em Credit_Limit:
Credit_Limit
34516.0    508
1438.3     507
9959.0      18
15987.0     18
23981.0     12
Name: count, dtype: int64

Diferença máxima entre Avg_Open_To_Buy e (Credit_Limit - Total_Revolving_Bal): 5.68e-14
-> Confirmado: Avg_Open_To_Buy é redundante e será removida.


In [5]:
df = df.drop(columns=[
    'CLIENTNUM',
    'Avg_Open_To_Buy',
    'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1',
    'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2'
], errors='ignore')

df = df.rename(columns={
    'Attrition_Flag': 'status_cliente',
    'Customer_Age': 'idade',
    'Gender': 'genero',
    'Dependent_count': 'dependentes',
    'Education_Level': 'escolaridade',
    'Marital_Status': 'estado_civil',
    'Income_Category': 'faixa_renda',
    'Card_Category': 'categoria_cartao',
    'Months_on_book': 'meses_como_cliente',
    'Total_Relationship_Count': 'total_produtos',
    'Months_Inactive_12_mon': 'meses_inativo_12m',
    'Contacts_Count_12_mon': 'contatos_12m',
    'Credit_Limit': 'limite_credito',
    'Total_Revolving_Bal': 'saldo_rotativo',
    'Total_Amt_Chng_Q4_Q1': 'var_valor_q4_q1',
    'Total_Trans_Amt': 'valor_total_transacoes',
    'Total_Trans_Ct': 'qtd_total_transacoes',
    'Total_Ct_Chng_Q4_Q1': 'var_qtd_q4_q1',
    'Avg_Utilization_Ratio': 'taxa_utilizacao_credito'
})

print("Colunas finais:", list(df.columns))

Colunas finais: ['status_cliente', 'idade', 'genero', 'dependentes', 'escolaridade', 'estado_civil', 'faixa_renda', 'categoria_cartao', 'meses_como_cliente', 'total_produtos', 'meses_inativo_12m', 'contatos_12m', 'limite_credito', 'saldo_rotativo', 'var_valor_q4_q1', 'valor_total_transacoes', 'qtd_total_transacoes', 'var_qtd_q4_q1', 'taxa_utilizacao_credito']


In [6]:
df["status_cliente"] = df["status_cliente"].replace({
    "Existing Customer": "Ativo",
    "Attrited Customer": "Cancelado"
})

df["escolaridade"] = df["escolaridade"].replace({
    "High School": "Ensino Médio",
    "Graduate": "Graduação",
    "Uneducated": "Sem escolaridade",
    "Post-Graduate": "Pós-graduação",
    "College": "Superior Incompleto",
    "Doctorate": "Doutorado",
    "Unknown": "Desconhecido"
})

df["estado_civil"] = df["estado_civil"].replace({
    "Married": "Casado",
    "Single": "Solteiro",
    "Divorced": "Divorciado",
    "Unknown": "Desconhecido"
})

df["faixa_renda"] = df["faixa_renda"].replace({
    "Less than $40K": "Menos de $40 mil",
    "$40K - $60K": "$40 mil - $60 mil",
    "$60K - $80K": "$60 mil - $80 mil",
    "$80K - $120K": "$80 mil - $120 mil",
    "$120K +": "Mais de $120 mil",
    "Unknown": "Desconhecido"
})

df["categoria_cartao"] = df["categoria_cartao"].replace({
    "Blue": "Azul",
    "Silver": "Prata",
    "Gold": "Ouro",
    "Platinum": "Platina"
})

df['genero'] = df['genero'].replace({'M': 'Masculino', 'F': 'Feminino'})

print(df[['escolaridade', 'estado_civil', 'faixa_renda']].apply(lambda s: s.value_counts()))

                     escolaridade  estado_civil  faixa_renda
$40 mil - $60 mil             NaN           NaN       1790.0
$60 mil - $80 mil             NaN           NaN       1402.0
$80 mil - $120 mil            NaN           NaN       1535.0
Casado                        NaN        4687.0          NaN
Desconhecido               1519.0         749.0       1112.0
Divorciado                    NaN         748.0          NaN
Doutorado                   451.0           NaN          NaN
Ensino Médio               2013.0           NaN          NaN
Graduação                  3128.0           NaN          NaN
Mais de $120 mil              NaN           NaN        727.0
Menos de $40 mil              NaN           NaN       3561.0
Pós-graduação               516.0           NaN          NaN
Sem escolaridade           1487.0           NaN          NaN
Solteiro                      NaN        3943.0          NaN
Superior Incompleto        1013.0           NaN          NaN


In [7]:
VALORES_LIMITE_SUSPEITOS = [1438.3, 34516.0]

df['limite_suspeito'] = df['limite_credito'].isin(VALORES_LIMITE_SUSPEITOS)

qtd_suspeitos = df['limite_suspeito'].sum()
print(f"Clientes com limite de crédito suspeito: {qtd_suspeitos} ({qtd_suspeitos/len(df)*100:.1f}%)")

Clientes com limite de crédito suspeito: 1015 (10.0%)


In [8]:
print("Formato final:", df.shape)
print("\nValores nulos por coluna:")
print(df.isnull().sum())

print("\nLinhas duplicadas:", df.duplicated().sum())

print("\nTipos de dados:")
print(df.dtypes)

print("\nResumo estatístico:")
df.describe()

Formato final: (10127, 20)

Valores nulos por coluna:
status_cliente             0
idade                      0
genero                     0
dependentes                0
escolaridade               0
estado_civil               0
faixa_renda                0
categoria_cartao           0
meses_como_cliente         0
total_produtos             0
meses_inativo_12m          0
contatos_12m               0
limite_credito             0
saldo_rotativo             0
var_valor_q4_q1            0
valor_total_transacoes     0
qtd_total_transacoes       0
var_qtd_q4_q1              0
taxa_utilizacao_credito    0
limite_suspeito            0
dtype: int64

Linhas duplicadas: 0

Tipos de dados:
status_cliente                 str
idade                        int64
genero                         str
dependentes                  int64
escolaridade                   str
estado_civil                   str
faixa_renda                    str
categoria_cartao               str
meses_como_cliente           int64

,idade,dependentes,meses_como_cliente,total_produtos,meses_inativo_12m,contatos_12m,limite_credito,saldo_rotativo,var_valor_q4_q1,valor_total_transacoes,qtd_total_transacoes,var_qtd_q4_q1,taxa_utilizacao_credito
count,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000
mean,46.325960,2.346203,35.928409,3.812580,2.341167,2.455317,8631.953698,1162.814061,0.759941,4404.086304,64.858695,0.712222,0.274894
std,8.016814,1.298908,7.986416,1.554408,1.010622,1.106225,9088.776650,814.987335,0.219207,3397.129254,23.472570,0.238086,0.275691
min,26.000000,0.000000,13.000000,1.000000,0.000000,0.000000,1438.300000,0.000000,0.000000,510.000000,10.000000,0.000000,0.000000
25%,41.000000,1.000000,31.000000,3.000000,2.000000,2.000000,2555.000000,359.000000,0.631000,2155.500000,45.000000,0.582000,0.023000
50%,46.000000,2.000000,36.000000,4.000000,2.000000,2.000000,4549.000000,1276.000000,0.736000,3899.000000,67.000000,0.702000,0.176000
75%,52.000000,3.000000,40.000000,5.000000,3.000000,3.000000,11067.500000,1784.000000,0.859000,4741.000000,81.000000,0.818000,0.503000
max,73.000000,5.000000,56.000000,6.000000,6.000000,6.000000,34516.000000,2517.000000,3.397000,18484.000000,139.000000,3.714000,0.999000


In [ ]:
print("Taxa de 'Desconhecido' em Faixa de Renda por Gênero:")
print(pd.crosstab(df['genero'], df['faixa_renda'] == 'Desconhecido', normalize='index') * 100)
# Achado: renda ausente em ~20% das mulheres vs. ~1% dos homens, e nenhuma
# mulher registrada na faixa "$120K +". Isso indica falha sistemática de
# coleta do dado original por gênero, não uma característica real da base.
# Recomendação: manter "Desconhecido" (não descartar), mas nunca cruzar
# faixa_renda x genero sem ressalva explícita nas análises.

Taxa de 'Desconhecido' em Faixa de Renda por Gênero:
faixa_renda      False      True 
genero                           
Feminino     80.216499  19.783501
Masculino    98.909625   1.090375


In [9]:
df = df.reset_index(drop=True)

In [10]:
os.makedirs('data/processed', exist_ok=True)

caminho_arquivo = 'data/processed/dataset_limpo.csv'
df.to_csv(caminho_arquivo, index=False)

print(f"Dataset salvo com sucesso em: {caminho_arquivo}")
print(f"Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")

Dataset salvo com sucesso em: data/processed/dataset_limpo.csv
Linhas: 10127 | Colunas: 20
